# Week 1 · How long is a sentence?
*MSc Digital Scholarship 2026*

In this notebook we will measure one of the oldest features in stylometry: **average sentence length**.
It sounds simple. It isn't — and working out *why* it isn't is the real lesson.

**By the end you should be able to:**
1. Run Python code in Colab and change it.
2. Turn a text into a list of sentences and a list of words.
3. Explain the *decisions* hidden inside a number like "average sentence length = 24.3 words".
4. Check code (yours, a classmate's, or an AI's) against an example you can verify by hand.

### How to use this notebook
- Click a grey code cell and press **Shift + Enter** to run it. Run cells **in order, from the top**.
- If things get confused: menu **Runtime → Restart session and run all**.
- You cannot break anything. Change the code, re-run it, see what happens.

### A rule for this course on generative AI
You are encouraged to use AI tools (ChatGPT, Claude, Gemini, Colab's built-in assistant…) to write and explain code.
But **you are responsible for every number you report.** Code that runs without errors is not the same as code that is right.
Throughout this notebook, look for the **Critical check** boxes.

## 1. A tiny text we can check by hand

Before running anything on a whole novel, we start with a few lines where *we already know the answer*.
This is the single most useful habit in computational text analysis: **test on something you can verify yourself.**

In [ ]:
sample = 'Mr. Bennet looked up from his book. "Is that you, my dear?" he asked. Mrs. Bennet did not reply... She had heard the news at 10 a.m. and could think of nothing else! The whole street knew by noon.'

print(sample)

> **Critical check — do this before reading on.**
> Count the sentences in `sample` yourself, by eye. Write your answer down.
> Now count the words in the first sentence. Is "10" a word? Is "a.m."?
> Did everyone in the room get the same answers?

### Some Python basics
- `sample` is a **variable**: a name for a piece of data. Here the data is a **string** (a piece of text).
- `print(...)` displays something.
- `len(...)` tells you how long something is. For a string, that's the number of characters.

In [ ]:
len(sample)

## 2. First attempt: split on full stops

A sentence ends with a full stop, so let's cut the text wherever there is a `.`

`sample.split(".")` gives us a **list** — an ordered collection of items, written with square brackets.

In [ ]:
attempt1 = sample.split(".")

print("Number of pieces:", len(attempt1))
attempt1

> **Critical check.** Compare this with your hand count.
> - What happened to `Mr.` and `Mrs.`?
> - What happened to the `...` and to `a.m.`?
> - What happened at the `?` and the `!`?
> - Why are there empty strings `''` in the list?
>
> This is exactly the kind of code a generative AI will happily give you if you ask a vague question —
> and it runs without any error message. **No error message ≠ correct.**

## 3. Second attempt: a pattern ("regular expression")

We can describe the end of a sentence more carefully:
*a `.`, `!` or `?`, followed by a space.*

Python's `re` module lets us describe patterns like this, in a mini-language called **regular expressions**. Here is our pattern, piece by piece:

| Piece | Meaning |
|---|---|
| `r'...'` | A **raw string**. Normally Python treats a backslash `\` as special inside a string (`\n` means "new line"). The `r` says "leave backslashes alone", so they reach the `re` module unchanged. Put an `r` in front of every pattern. |
| `[.!?]` | Square brackets mean **any one** of these characters: a full stop, an exclamation mark or a question mark. |
| `(?<=...)` | "**just after** this". It checks what comes before, but doesn't include it in the cut, so the punctuation stays with its sentence. |
| `\s` | Any **whitespace** character: a space, a tab, or a line break. |
| `+` | **One or more** of the thing before it, so `\s+` means "one or more spaces or line breaks". |

Put together, `r'(?<=[.!?])\s+'` reads: **"cut at any run of whitespace that comes just after a `.`, `!` or `?`"**.

You don't need to memorise the notation, but you should be able to *test* what it does.

In [ ]:
import re

attempt2 = re.split(r'(?<=[.!?])\s+', sample)

print("Number of sentences:", len(attempt2))
for s in attempt2:
    print("→", s)

**Experiment.** What does each piece of the pattern do? Change the pattern in the cell below and predict the result before you run it:
- `r'\s+'` (just the whitespace part)
- `r'[.!?]\s+'` (without the "just after")
- `r'(?<=[.!])\s+'` (without the `?`)

An AI assistant can explain a regular expression for you. Ask it to explain this one, then check its explanation by experimenting here.
Websites such as regex101.com also show what a pattern matches, step by step.

In [ ]:
pattern = r'[.!?]\s+'

re.split(pattern, sample)

> **Critical check.**
> - Which problems from Attempt 1 has this fixed?
> - Which remain? Which are *new*?
> - Can you think of a sentence from your own field (a quotation, a citation like "p. 42", "St. Paul", "c. 1600") that would fool this pattern?

## 4. Third attempt: a ready-made sentence splitter (NLTK)

The **Natural Language Toolkit (NLTK)** is a widely used Python library for text analysis.
Its sentence splitter (called *Punkt*) was **trained on a large collection of English text** to learn
which full stops usually end sentences and which belong to abbreviations like "Mr."

That makes it better on average — but notice: it is now a *model* with its own assumptions (which English? which period? which genres?),
not a rule you can read.

In [ ]:
import nltk
nltk.download('punkt', quiet=True)       # the splitter's trained data
nltk.download('punkt_tab', quiet=True)   # (newer versions of NLTK need this too)

from nltk.tokenize import sent_tokenize

attempt3 = sent_tokenize(sample)

print("Number of sentences:", len(attempt3))
for s in attempt3:
    print("→", s)

> **Critical check.** Is this now "correct"? Compare it once more with your hand count.
> Look closely at `he asked.` and at the `...` — what has the splitter decided, and do you agree?
> Is there a single correct answer for `Mrs. Bennet did not reply... She had heard…`? (Is "..." the end of a sentence?)
> Decisions like this are **interpretive**, not technical. Your methods section should say which choice you made.

## 5. What is a word?

To get *average sentence length* we also need to count words. Again we must decide what counts.

The pattern below finds runs of letters, allowing an apostrophe or hyphen inside a word
(so `don't` and `well-known` each count as one word).

In [ ]:
def words_in(text):
    # letters, optionally joined by an apostrophe or hyphen: don't, well-known, Bennet's
    return re.findall(r"[A-Za-z]+(?:['’-][A-Za-z]+)*", text)

words_in('"Is that you, my dear?" he asked at 10 a.m. -- she didn\'t hear the well-known bell.')

> **Critical check.** What happened to `10`? To `a.m.`? To `--`?
> Would these choices matter more for a Victorian novel, a tweet, or a medieval charter?

> **A note on Project Gutenberg's italics.** Plain text can't show italics, so Gutenberg marks them with underscores: `_you_`, `_Pride and Prejudice_`.
> Our `words_in` pattern only collects letters, so `_you_` simply becomes `you`. That's convenient, but notice what we lose: in *Pride and Prejudice*
> the words Austen italicises most often for emphasis are *that*, *you*, *her*, *I* and *me*.
> Other code might treat `_you_` as a different word from `you`. (Python's `\w`, used in many AI answers, counts `_` as a letter.)
> Which is right depends on your question: is emphasis part of what you want to measure?

### Characters are numbers

To a computer, every character is a number. **Unicode** is the agreed list that gives each character its number: Latin letters,
accented letters, Greek, Hebrew, the long s (ſ) of early printed books, and punctuation such as the curly apostrophe `’`.
The function `ord` tells you a character's number.

In [ ]:
straight = "'"
curly = "’"

print(ord(straight), ord(curly))
print(straight == curly)

They look almost the same to us, but to Python they are different characters. So these two words are different strings,
and a word count would treat them as two different words:

In [ ]:
words_in("don't don’t")

In [ ]:
set(words_in("don't don’t"))      # a set keeps only the distinct items

### UTF-8: how the numbers are stored
A file stores **bytes**, and each byte holds a number from 0 to 255. **UTF-8** is the most common way of writing Unicode characters as bytes:
ordinary English letters take one byte, other characters take two, three or four.

In [ ]:
for ch in ["a", "é", "’", "ſ", "α"]:
    print(ch, ord(ch), ch.encode("utf-8"))

When we download a book later in this notebook, the line `.decode("utf-8")` turns the bytes back into characters.
If a program guesses the wrong encoding, you get garbled text. You will meet this in old web pages, spreadsheets and archive exports,
so it's worth recognising:

In [ ]:
data = "Bennet’s".encode("utf-8")

print(data.decode("utf-8"))      # the right encoding
print(data.decode("cp1252"))     # a wrong guess (an old Windows encoding)

### The same letter, stored two ways
An accented letter such as *é* can be stored as one character, or as *e* followed by a separate accent mark.
They look identical, but they don't match until the text is **normalised**.

In [ ]:
import unicodedata

one = "café"                                   # é as a single character
two = unicodedata.normalize("NFD", one)        # e followed by a separate accent

print(one, two)
print(one == two, len(one), len(two))
print(unicodedata.normalize("NFC", two) == one)   # normalise, and they match again

> **Critical check.**
> - The AI's word-counting code in class looked for the straight apostrophe `'`. *Pride and Prejudice* on Gutenberg uses only the curly one `’`.
>   What happened to "don’t" and "Bennet’s"? (After Section 7, try `text.count("'")` and `text.count("’")`.)
> - Our `words_in` pattern accepts both kinds of apostrophe. Look back at it in Section 5: can you see where?
> - Think about a text from your own field. Does it use curly or straight quotes, accented letters, old letter forms, or another alphabet?
>   Who decided how it was typed or transcribed, and would your code find every form of a word?

## 6. Putting it together: a function

A **function** packages up some steps so we can reuse them. We give it a name (`average_sentence_length`),
some inputs (the text, and which sentence-splitter to use), and it gives back (`return`s) an answer.

In [ ]:
def split_on_full_stops(text):
    return [s for s in text.split(".") if s.strip()]      # drop the empty pieces

def split_with_pattern(text):
    return re.split(r'(?<=[.!?])\s+', text.strip())

def split_with_nltk(text):
    return sent_tokenize(text)


def average_sentence_length(text, splitter):
    sentences = splitter(text)
    lengths = [len(words_in(s)) for s in sentences]
    return sum(lengths) / len(lengths)


for splitter in [split_on_full_stops, split_with_pattern, split_with_nltk]:
    print(f"{splitter.__name__:22}  {average_sentence_length(sample, splitter):.2f} words per sentence")

> **Critical check.** Using your hand counts from Section 1, work out the average sentence length on a calculator.
> Which method agrees with you? Same text, three defensible-looking methods — how many different answers?
> If a method gets the *right number* but splits the sentences differently from you, is it right? (Right answer, wrong reasons is still wrong.)

## 7. A real text: *Pride and Prejudice*

Now for a whole novel. [Project Gutenberg](https://www.gutenberg.org) provides free plain-text editions of out-of-copyright books.
Each book has a number — *Pride and Prejudice* is **1342**.

In [ ]:
import urllib.request

def get_gutenberg(book_id):
    url = f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt"
    raw = urllib.request.urlopen(url).read().decode("utf-8")
    return raw.replace("\r\n", "\n")      # Windows-style line endings -> plain newlines

raw = get_gutenberg(1342)

print(len(raw), "characters")
print(raw[:1500])

> **Critical check.** Look at what we downloaded. Is all of it Jane Austen?
> Scroll to the end too (next cell). If we analyse `raw` as it is, what will we actually be measuring?

In [ ]:
print(raw[-1500:])

Gutenberg wraps every book in a licence and front matter. Fortunately they mark where the book proper starts and ends
with lines beginning `*** START OF` and `*** END OF`. We keep only what lies between them.

In [ ]:
def strip_gutenberg(raw):
    start = raw.find("*** START OF")
    start = raw.find("\n", start) + 1        # move to the line after the marker
    end = raw.find("*** END OF")
    return raw[start:end].strip()

text = strip_gutenberg(raw)

print(len(text), "characters after stripping")
print(text[:1500])

> **Critical check.** Is the beginning *now* pure Austen? (Hint: look for a preface, illustrations notes, a table of contents, chapter headings.)
> Perfect cleaning is often impossible; what matters is that you **know** what's left in and **say so**.

## 8. Does the method matter on a whole novel?

We put the results into a small table using **pandas**, the standard Python library for tables of data.

In [ ]:
import pandas as pd

rows = []
for splitter in [split_on_full_stops, split_with_pattern, split_with_nltk]:
    sentences = splitter(text)
    rows.append({
        "method": splitter.__name__,
        "sentences": len(sentences),
        "average length (words)": round(average_sentence_length(text, splitter), 2),
    })

pd.DataFrame(rows)

> **Discussion.** How big are the differences? Big enough to change a conclusion if you were comparing two authors?
> If a paper reports "Austen's mean sentence length is X", what would you need to know before you could compare it with your own figure?

## 9. Beyond the average: look at the actual sentences

An average is a summary — it hides a lot. From now on we use the NLTK splitter.
Let's keep *every* sentence and its length, and then **look at the extremes**. The extremes are where errors hide.

In [ ]:
sentences = [" ".join(s.split()) for s in sent_tokenize(text)]   # tidy line breaks into single spaces
lengths = [len(words_in(s)) for s in sentences]

table = pd.DataFrame({"sentence": sentences, "words": lengths})
table.describe()

In the summary above, `mean` is the average, `50%` is the **median** (the middle value), and `std` is the
**standard deviation** (how spread out the lengths are). Why might the mean and median differ?

In [ ]:
# The ten shortest "sentences"
pd.set_option("display.max_colwidth", 200)
table.sort_values("words").head(10)

> **Critical check.** Are these really sentences? Where do they come from? (Chapter headings? Dialogue? Something else?)
> Look for any that start with an opening quotation mark followed by `Mr.` — what has gone wrong there, given what Section 4 told you about how NLTK recognises abbreviations?
> Is there a "sentence" with **zero** words? What is it?
> Should they be removed — and if so, would removing them be *cleaning* the data or *changing* it?

In [ ]:
# The five longest sentences
table.sort_values("words", ascending=False).head(5)

> **Critical check.** Read these. **Did Jane Austen write all of them?** (Remember the front matter in Section 7.)
> If not, then our figure for "Austen's sentence length" is partly someone else's — and the longest sentences pull the mean upwards most of all.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 4))
plt.hist(lengths, bins=range(0, 120, 2), color="steelblue")
plt.xlabel("Sentence length (words)")
plt.ylabel("Number of sentences")
plt.title("Pride and Prejudice: sentence lengths")
plt.show()

## 10. Your turn

Try these in pairs. Use an AI assistant if you like — but complete the **critical check** for each.

1. **Another novel.** Change `1342` to another Gutenberg number — e.g. *Emma* `158`, *A Tale of Two Cities* `98`,
   *Frankenstein* `84` — and compute its average sentence length. Is the start/end stripping still correct for that book?
2. **Median instead of mean.** Change `average_sentence_length` so it returns the median.
   (Hint: `import statistics` then `statistics.median(lengths)`.) Does the ranking of authors change?
3. **Your own text.** Upload a `.txt` file using the cell below and analyse it: change `UPLOAD = False` to `UPLOAD = True` first. (It's switched off so that **Runtime → Run all** doesn't stop and wait for a file.) Only use a public-domain text or your own writing that you'd be happy to make public (see *Your data and Colab* at the top of the warm-up notebook).
4. **Break it.** Write a short sample text (like the one in Section 1) designed to fool NLTK's splitter. Share the best one with the class.

In [ ]:
# Optional: upload your own plain-text file (only works in Google Colab)
# Only public-domain text, or your own writing you'd be happy to make public
UPLOAD = False      # change to True, then run this cell, to choose a file

if UPLOAD:
    from google.colab import files
    uploaded = files.upload()
    my_text = next(iter(uploaded.values())).decode("utf-8")
    print(round(average_sentence_length(my_text, split_with_nltk), 2), "words per sentence")
else:
    print("Upload is switched off. Set UPLOAD = True to choose a file.")

## 11. Using generative AI critically: an exercise

This is the same routine as the most-common-words exercise, with a new question. Ask an AI assistant:

> Write Python code for Google Colab that downloads *Pride and Prejudice* from Project Gutenberg and calculates the average sentence length.

Paste its code into a new cell (**Insert → Code cell**), run it, and work through this checklist:

| Question | Why it matters |
|---|---|
| Does it run? | The lowest bar, not the highest. |
| What average does it give? How does that compare with our three methods in Section 8? | Same novel, another answer. |
| What does it give for our `sample`? (You may need to change one line so it uses `sample`.) Does that match your hand count? | Verifiable test case. |
| How does it decide where a sentence ends? What does it do with "Mr."? | Hidden definition of *sentence*. |
| How does it decide what a word is? Does it count punctuation as words? | Hidden definition of *word*. |
| Did it strip the Gutenberg licence? | Are we measuring the right text? |
| Ask it to explain each line. Does the explanation match what the code actually does? | AI explanations can be confidently wrong. |
| Ask it the same question again in a new chat. Same code? Same answer? | Reproducibility. |

Write two or three sentences describing what you found. **This kind of reflection is what we'll be looking for in your assessed work.**

## 12. Class results: one prompt, many answers

Everyone in the class gave an AI the same prompt. Here we collect the average sentence length that each AI's code reported for *Pride and Prejudice*,
and plot them against our own three methods from Section 8.

Enter one line per result in the cell below: which AI, and the number its code reported. Or, if your lecturer has set up a shared sheet,
paste its link (published as CSV, with columns `model` and `average`).

In [ ]:
class_results = [
    # {"model": "ChatGPT", "average": 20.1},      <- one line like this per result (these numbers are made up)
    # {"model": "Claude", "average": 21.5},
]

CLASS_SHEET_URL = ""      # or a published CSV link with columns: model, average

if CLASS_SHEET_URL:
    class_results = pd.read_csv(CLASS_SHEET_URL).to_dict("records")

results = pd.DataFrame(class_results, columns=["model", "average"])
results

The function below draws one row per AI, with a dot for each result. The dashed lines are our three methods from Section 8.
We'll use the same function again next week for burstiness.

In [ ]:
def plot_class_results(results, measure, references, xlabel):
    if results.empty:
        print("No results yet: add some to class_results above, then run that cell and this one again.")
        return
    models = sorted(results["model"].unique())
    fig, ax = plt.subplots(figsize=(9, 1.5 + 0.6 * len(models)))
    for i, model in enumerate(models):
        values = results.loc[results["model"] == model, measure]
        ax.scatter(values, [i] * len(values), s=80, color="steelblue", edgecolor="white", linewidth=1.5, zorder=3)
    for label, value in references.items():
        ax.axvline(value, color="dimgray", linestyle="--", linewidth=1, zorder=1)
        ax.text(value, len(models) - 0.45, f"{label} ", rotation=90, va="bottom", ha="right", fontsize=9, color="dimgray")
    ax.set_yticks(range(len(models)))
    ax.set_yticklabels(models)
    ax.set_ylim(-0.6, len(models) + 1.0)
    ax.set_xlabel(xlabel)
    ax.grid(axis="x", color="lightgray", linewidth=0.5)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    plt.tight_layout()
    plt.show()


short_names = {"split_on_full_stops": "full stops", "split_with_pattern": "pattern", "split_with_nltk": "NLTK"}
our_methods = {short_names[r["method"]]: r["average length (words)"] for r in rows}
plot_class_results(results, "average", our_methods, "Average sentence length of Pride and Prejudice (words)")

> **Discussion.**
> - How far apart are the AIs' answers? How far apart are our own three methods? Which spread is bigger?
> - Did the same AI give the same answer to different people?
> - Pick the result furthest from the others and look at its code. Can you find the decision that explains it: the licence, the sentence splitter, what counts as a word?
> - If you had to report one number for Austen's average sentence length, which would it be, and what would you write in your methods to justify it?

## Looking ahead: burstiness

Two texts can have the *same* average sentence length but feel completely different:
one steady and even, the other alternating short punchy sentences with long winding ones.
The **standard deviation** (`std` above) starts to capture this. Next time we'll turn it into a **burstiness** measure
and ask whether it can help tell authors apart — or tell human writing from AI-generated text.

In [ ]:
import statistics
print("mean:", round(statistics.mean(lengths), 2))
print("standard deviation:", round(statistics.stdev(lengths), 2))